# GPT Action 库（中间件）：Azure Function

## 简介

本页面为构建中间件以将 GPT Action 连接到特定应用的开发者提供说明和指南。在继续之前，请先熟悉以下内容： 
- [GPT Actions 简介](https://platform.openai.com/docs/actions)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

这个 GPT Action 概述了如何构建 **Azure Function**（微软基于云的函数构建服务）。本文档帮助用户设置一个受 OAuth 保护的 Azure Function，用于连接 GPT Action 以及一个示例应用。 

### 价值 + 业务用例示例

**价值**：用户现在可以利用 ChatGPT 的自然语言能力直接连接到 Azure Function。这可以体现在以下几个方面：

- GPT Actions 中有 10 万字符的限制：用户可以使用中间件对 API 返回的文本响应进行预处理。例如，你可以在中间件中使用 OpenAI 的 API 先对文本进行摘要，再将其发回 ChatGPT。
- 通常对于 actions，用户依赖 SaaS API 返回文本。你可以将供应商 API 的响应转换为易于理解的文本，并且它可以处理结构化和非结构化数据等不同的数据类型。
- 它可以返回文件，而不仅仅是文本。这对于提供 CSV 文件用于数据分析、或返回 PDF 文件很有用，ChatGPT 会将其视为上传的文件。 


**用例示例**： 
- 用户需要在 Sharepoint 中查找文件，但需要在 ChatGPT 和 Sharepoint 之间有一个中间件应用
- 用户在 Azure function 中构建了一系列连续步骤，并需要能够通过 ChatGPT 启动这一流程

## 应用信息

### 应用关键链接

开始之前，请先查看该应用的以下链接：
- 应用网站：https://learn.microsoft.com/en-us/azure/azure-functions/
- 应用 API 文档：https://learn.microsoft.com/en-us/azure/azure-functions/functions-reference/

### 应用前提条件

开始之前，请确保在你的应用环境中完成以下步骤：
- 拥有可以创建 Azure Function Apps 和 Azure Entra 应用注册的 Azure 门户访问权限

## 应用设置

### 安装应用

你可以在[这里](https://learn.microsoft.com/en-us/azure/azure-functions/functions-overview?pivots=programming-language-csharp)文档的左侧了解 Azure Functions 支持的语言和部署选项。 

#### 选项 1：使用 VSCode

关于如何使用 VSCode 进行部署，请参阅 Microsoft 的文档[这里](https://learn.microsoft.com/en-us/azure/azure-functions/functions-develop-vs-code?tabs=node-v4,python-v2,isolated-process\&pivots=programming-language-javascript)。如果你熟悉这种方式，可以放心使用。 


#### 选项 2：直接在 Azure 门户中操作

关于如何使用 Azure 门户进行部署，请参阅[这里](https://learn.microsoft.com/en-us/azure/azure-functions/functions-create-function-app-portal?pivots=programming-language-javascript)的文档。我们将在这里逐步演示一个示例。


##### 第 1 部分：创建函数

![](../../../images/create_function_app.png)

1. 创建一个 [Azure Function app](https://learn.microsoft.com/en-us/azure/azure-functions/functions-overview?pivots=programming-language-csharp)。我使用了以下设置，但你可以使用任何你熟悉的设置。注意，并非所有语言/操作系统组合都允许直接在控制台中编辑函数——我下面选择的组合是允许的。在我的演练中，我将所有设置保留为默认值，只做了下面的选择。以下设置可直接用于[这里](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_action_sharepoint_doc)和[这里](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_action_sharepoint_text)的 SharePoint Node.js 解决方案。 

   1. 基本信息

      1. _是要部署代码还是容器镜像？：_  **Code**

      2. _运行时堆栈：_ **Node.js**

      3. _操作系统：_ **Windows**

   2. 网络

      1. _启用公共访问_：**开启（需要开启才能连接到 GPT）**

2. 完成上述操作后，你会进入“Deployments”页面。部署完成后（应该只需几分钟），点击 **“Go to Resource”** 返回 Function App

  > 第一次尝试时你可能会遇到错误，再次点击创建，大概率就能成功。 

##### 第 2 部分：设置身份验证

3. 在 Azure Function App 左侧菜单中，点击 **Settings** 菜单下的 **Authentication**。 

   1. 添加身份提供程序

   2. 选择 **Microsoft** 作为身份提供程序。 

   3. 租户类型选择 **Workforce**

   4. **创建一个新应用。** 如果使用现有应用，步骤也大致相同，但创建新应用会更容易，因为它会通过“Easy Auth”自动配置回调 URL 并公开 API。你可以在[**这里**](https://learn.microsoft.com/en-us/azure/app-service/overview-authentication-authorization)了解更多相关信息。

   5. 将此页面上的其他所有设置保留为默认值，也可以根据你内部的规范进行调整。

   6. 在 **permissions** 标签页中，点击 **Add Permission**，添加 **Files.Read.All** 和 **Sites.ReadAll**，然后点击 **Add**。这允许该应用读取文件，这对于使用 Microsoft Graph Search API 很重要。如果你不是将其用于[这里](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_action_sharepoint_doc)和[这里](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_action_sharepoint_text)的 SharePoint 解决方案，可以跳过此步。

4. 创建完成后，**点击你刚刚创建的企业应用**（即离开 Function App 页面，进入你刚刚创建的 Enterprise Application）**。** 现在我们要再给它授予一项权限：通过模拟登录该应用的用户来执行 Azure Function。更多详情请参阅[这里](https://learn.microsoft.com/en-us/azure/app-service/configure-authentication-provider-aad?tabs=workforce-tenant)。

   1. 在主页面上，点击“**View API Permissions”**

   2. 在 **APIs my organization uses** 中搜索 **Microsoft Azure App Service**，找到 **user\_impersonation** 

   3. 添加它，然后需要 Azure 门户的管理员执行 **Grant Admin Consent**。

5) **在该企业应用中**，点击左侧菜单 **Manage** 下的 **“Expose an API”**，然后使用 **Copy to Clipboard** 按钮复制已创建的 **scope**。scope 应类似于“api://\<insert-uuid>/user\_impersonation”。**将其保存下来，稍后作为** `SCOPE` **使用。**

6) 点击左侧菜单 **Manage** 下的 **“Authentication”**

   1. 在 **Web** 部分下，你会注意到已经自动添加了一个回调 URI。为了测试，再添加 Postman 的重定向 URI（<https://oauth.pstmn.io/v1/callback>）。

7) 在左侧进入 **Overview**。复制 **application (client) ID** 和 **directory (tenant) ID**，**并保存下来，稍后分别作为** `CLIENT_ID` **和** `TENANT_ID` **使用。**

##### 第 3 部分：设置测试函数

8. 返回首页离开此页面，然后回到你的 **Function App**。

9. 点击 **Create Function**。在本示例中，我将在门户中进行开发，但你也可以使用 VSCode 或其他 IDE。

   1. 选择 **HTTP trigger**

   2. 对于 **Authorization Level**，你可以选择任意密钥类型。

      1. 注意，第一次操作可能会报错，但函数很可能已经创建成功，刷新页面查看即可。

10. 点击你刚刚创建的函数（可能需要点击刷新才能看到）。点击 **Get Function URL** 并保存下来，以便在 Postman 中测试。稍后在 GPT 中创建 OpenAPI 规范时也会用到它。 

![](../../../images/get_function_url.png)

11. 回到 function app，点击 **Configuration**。显示 `MICROSOFT_PROVIDER_AUTHENTICATION_SECRET` 变量的值，复制它（点击 advanced edit 进行复制），并**保存下来供稍后使用**。  

此时，你应该已经创建了一个测试函数，并保存了 **client id、tenant id、secret、scope 和函数 URL**。现在可以在 Postman 中测试身份验证了

##### 第 4 部分：在 Postman 中测试身份验证

12. 在 Postman 中使用以下 OAuth 设置尝试访问你创建的端点：

    1. **Grant Type：** Authorization Code

    2. **Auth URL**：https://login.microsoftonline.com/`TENANT_ID`/oauth2/v2.0/authorize

    3. **Auth Token URL**：https://login.microsoftonline.com/`TENANT_ID`/oauth2/v2.0/token

    4. **Client ID：** 上面第 7 步中的 `CLIENT_ID`

    5. **Client secret：** 上面第 11 步中的 `MICROSOFT_PROVIDER_AUTHENTICATION_SECRET `

    6. **Scope**：上面第 5 步中的 `SCOPE`

    7. **Client credentials**：Send client credentials in body

13. 你需要点击 **Get New Access Token**，然后访问你在上面第 10 步中保存的端点。如果成功，你应该会得到以下响应：`”This HTTP triggered function executed successfully. Pass a name in the query string or in the request body for a personalized response.”`

##### 第 5 部分：在 Azure Function 上设置你的应用

这一步需要单独完成，并且取决于你的具体应用。示例请参阅 [SharePoint Cookbook](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_action_sharepoint_doc)。 

##### 第 6 部分：设置 ChatGPT

14. 为你的端点生成 OpenAPI 规范。 

15. 将其粘贴到 GPT 的 Actions 部分，并选择 OAuth 作为身份验证类型。按照上面在 Postman 中的方式填写 OAuth 设置。 

16. 保存 action 后，你会在 GPT 配置的底部看到一个回调 URI。复制该 URL，然后**回到 Azure 门户中的 Function App**。

17. 点击 **Settings** 下的 **Authentication**，然后点击你的 Entra 应用。

18. 进入之后，点击 **Manage** 部分下的 **Authentication**。

19. 在该页面的 **Web** 部分下添加一个新的重定向 URI，粘贴你在第 16 步中获得的回调 URI，然后点击保存。 

20. 测试 GPT，它应该能按预期工作。

## ChatGPT 步骤

### 自定义 GPT 指令 

*这部分因应用而异。示例请参阅 [SharePoint Cookbook](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_action_sharepoint_doc)*

### OpenAPI Schema 

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步的工作原理。

下面是连接到此中间件的一个示例。你需要在此部分填入你的应用和函数的信息。 

In [ ]:
openapi: 3.1.0
info:
  title: {insert title}
  description: {insert description}
  version: 1.0.0
servers:
  - url: https://{your_function_app_name}.azurewebsites.net/api
    description: {insert description}
paths:
  /{your_function_name}?code={enter your specific endpoint id here}:
    post:
      operationId: {insert operationID}
      summary: {insert summary}
      requestBody: 
{the rest of this is specific to your application}

## 身份验证说明

以下是为此第三方应用设置身份验证的说明。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，更详细地了解这一步的工作原理。

### Action 前置步骤

在 ChatGPT 中设置身份验证之前，请先在应用中完成以下步骤。

*按照上面的第 2 和第 4 步设置身份验证*

### 在 ChatGPT 中

在 ChatGPT 中，点击“Authentication”并选择 **“OAuth”**。填写以下信息。 

- **Client ID**：*参见上面第 12 步*
- **Client Secret**：*同上*
- **Authorization URL**：*同上*
- **Token URL**：*同上*
- **Scope**：*同上*
- **Token**：*同上*

### Action 后续步骤

在 ChatGPT 中设置好身份验证后，请在应用中按照以下步骤完成 Action 的配置。 

*测试此应用的方法请参见上文*

*有没有希望我们优先支持的集成？我们的集成中有错误吗？请在我们的 GitHub 中提交 PR 或 issue，我们会尽快查看。*